In [ ]:
import re 
import json
import os
import sys
from functools import reduce
from collections import Counter
import numpy as np
import ast
from copy import deepcopy
from typing import List, Dict, Union
from collections import Counter
import joblib

PAI_ROOT_PATH = "../../.."
sys.path.insert(0, PAI_ROOT_PATH)

# useful link : https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

In [ ]:
def calculate_statistics(values: List[Union[float, int]]) -> Dict[str, Union[float, int]]:
    return {
        'count': len(values),
        'max': round(max(values),3),
        'mean': round(float(np.mean(values)), 3),
        'median': round(float(np.median(values)), 3),
        'min': round(min(values),3),
        'std': round(float(np.std(values)), 3),
        'sum': round(sum(values),3)
    }

In [ ]:
BENCHMARK = 'musique' # TO CHANGE
EXP_NAME = 'musique_qwen257b_medium_mixture(#2cc1dbd0)(v2.3.7)' # TO CHANGE
KNOWLEDGE_GRAPH_NAME = 'qwen257b_050226_v2prompts' # TO CHANGE

LOAD_MEDIUM_REASEONER_LOG_PATH = f"{PAI_ROOT_PATH}/../pai2_mainexps/{BENCHMARK}/{EXP_NAME}/log/qa/kg_reasoner/medium/main/.log" # TO CHANGE
LOAD_QUESTION_TO_RELEVANTIDS_PATH = f"{PAI_ROOT_PATH}/../pai2_knowledge_graphs/{BENCHMARK}/{KNOWLEDGE_GRAPH_NAME}/matched_tnids_to_questions.json"

SAVE_MATCHED_NODES_PATH = f"artifacts/{BENCHMARK}/{EXP_NAME}/matched_nodes_info.json"
SAVE_NODEMATCHING_SCORES_PATH = f"artifacts/{BENCHMARK}/{EXP_NAME}/node_matching_scores.json"

In [ ]:
with open(LOAD_MEDIUM_REASEONER_LOG_PATH, 'r', encoding='utf-8') as fd:
    RAW_LOG = fd.read()

In [ ]:
with open(LOAD_QUESTION_TO_RELEVANTIDS_PATH, 'r', encoding='utf-8') as fd:
    MATCHED_QUESTIONS_TO_RELEVANTTIDS = json.loads(fd.read())

### PAI-1

### PAI-2

In [ ]:
START_QUERY_HANDLING_LINE = "START MEDIUM KG-REASONING..."
START_ENTITIES_EXTRACTION_LINE = "STAGE#2.1.1 - ENTITIES EXTRACTION"
START_E2N_MATCHING_LINE = "STAGE#2.1.2 - ENTITIES-TO-KGOBJECTS MATCHING"

In [ ]:
MAIN_LOG_POINTS = []
LOG_LINES = RAW_LOG.split("\n")
for idx, log_line in enumerate(LOG_LINES):
    if log_line.endswith(START_QUERY_HANDLING_LINE):
        query_hash = LOG_LINES[idx+1].split(" ")[-2]
        query = LOG_LINES[idx+2].split("Question: ")[1].rstrip("\n. ")
        MAIN_LOG_POINTS.append((idx, 'start_query_handling', query_hash, query))
    elif log_line.endswith(START_ENTITIES_EXTRACTION_LINE):
        extracted_entities = ast.literal_eval(LOG_LINES[idx+2].split("RESULT: ")[1].strip("\n. "))
        MAIN_LOG_POINTS.append((idx, 'start_entities_extraction', extracted_entities))
    elif log_line.endswith(START_E2N_MATCHING_LINE):
        num_entities = len(MAIN_LOG_POINTS[-1][2])
        matched_nodes = []
        for entity_idx in range(num_entities):
            matched_nodes.append(LOG_LINES[idx+3+entity_idx].split("] ")[1].split(", "))
        MAIN_LOG_POINTS.append((idx, 'matched_object_nodes', matched_nodes))

In [ ]:
MATCHED_NODES_PER_QUESTION_INFO = dict()

cur_question_info = dict()
cur_question_hash = None 
cur_matched_nodes_info = None
for point_info in MAIN_LOG_POINTS:
    if point_info[1] == 'start_query_handling':
        if cur_question_hash is not None:
            MATCHED_NODES_PER_QUESTION_INFO[cur_question_hash] = deepcopy(cur_question_info)

        cur_question_hash = point_info[2]
        cur_question_info = {'question': point_info[3], 'search_steps': []}

    elif point_info[1] == 'start_entities_extraction':
        cur_matched_nodes_info = dict()
        cur_matched_nodes_info['entities'] = point_info[2]

    elif point_info[1] == 'matched_object_nodes':
        cur_matched_nodes_info['matched_object_nodes'] = point_info[2]
        cur_question_info['search_steps'].append(deepcopy(cur_matched_nodes_info))
        cur_matched_nodes_info = None

MATCHED_NODES_PER_QUESTION_INFO[cur_question_hash] = deepcopy(cur_question_info)

In [ ]:
len(MATCHED_NODES_PER_QUESTION_INFO)

In [ ]:
with open(SAVE_MATCHED_NODES_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(MATCHED_NODES_PER_QUESTION_INFO, ensure_ascii=False, indent=2))

In [ ]:
# sanity check
len(set(MATCHED_QUESTIONS_TO_RELEVANTTIDS.keys()).intersection(set(MATCHED_NODES_PER_QUESTION_INFO.keys())))

In [ ]:
#MATCHED_QUESTIONS_TO_RELEVANTTIDS['5d630ab3267d560b99884eb2530a6463']

In [ ]:
#MATCHED_NODES_PER_QUESTION_INFO['5d630ab3267d560b99884eb2530a6463']

In [ ]:
# calculating Precision, Recall and F1 for matched object nodes that are aggregated by entity, search step and question

RAW_QLEVEL_SCORES = {'precision': [], 'recall': []}
RAW_SLEVEL_SCORES = {'precision': [], 'recall': []}
RAW_ELEVEL_SCORES = {'precision': [], 'recall': []}

for question_hash, question_info in MATCHED_NODES_PER_QUESTION_INFO.items():
    unique_relevant_qlevel_object_nodes = set()
    unique_irrelevant_qlevel_object_nodes = set()
    for ss_info in question_info['search_steps']:
        unique_relevant_slevel_object_nodes = set()
        unique_irrelevant_slevel_object_nodes = set()
        for matched_object_nodes in ss_info['matched_object_nodes']:
            unique_relevant_elevel_object_nodes = set()
            for object_node in matched_object_nodes:
                if object_node in MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_object_nodes_info'].keys():
                    unique_relevant_elevel_object_nodes.add(object_node)
                else:
                    unique_irrelevant_slevel_object_nodes.add(object_node)
            
            elevel_precision = len(unique_relevant_elevel_object_nodes) / len(matched_object_nodes)
            elevel_recall = len(unique_relevant_elevel_object_nodes) / len(MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_object_nodes_info'])
            #elevel_f1 = (2 * elevel_precision * elevel_recall) / (elevel_precision + elevel_recall)
            RAW_ELEVEL_SCORES['precision'].append(elevel_precision)
            RAW_ELEVEL_SCORES['recall'].append(elevel_recall)
            #RAW_ELEVEL_SCORES['f1'].append(elevel_f1)

            unique_relevant_slevel_object_nodes.update(unique_relevant_elevel_object_nodes)

        slevel_precision = len(unique_relevant_slevel_object_nodes) / (len(unique_irrelevant_slevel_object_nodes) + len(unique_relevant_slevel_object_nodes))
        slevel_recall = len(unique_relevant_slevel_object_nodes) / len(MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_object_nodes_info'])
        #slevel_f1 = (2 * slevel_precision * slevel_recall) / (slevel_precision + slevel_recall)
        RAW_SLEVEL_SCORES['precision'].append(slevel_precision)
        RAW_SLEVEL_SCORES['recall'].append(slevel_recall)
        #RAW_SLEVEL_SCORES['f1'].append(slevel_f1)

        unique_irrelevant_qlevel_object_nodes.update(unique_irrelevant_slevel_object_nodes)
        unique_relevant_qlevel_object_nodes.update(unique_relevant_slevel_object_nodes)

    qlevel_precision = len(unique_relevant_qlevel_object_nodes) / (len(unique_irrelevant_qlevel_object_nodes) + len(unique_relevant_qlevel_object_nodes))
    qlevel_recall = len(unique_relevant_qlevel_object_nodes) / len(MATCHED_QUESTIONS_TO_RELEVANTTIDS[question_hash]['relevant_unique_object_nodes_info'])
    #qlevel_f1 = (2 * qlevel_precision * qlevel_recall) / (qlevel_precision + qlevel_recall)
    RAW_QLEVEL_SCORES['precision'].append(qlevel_precision)
    RAW_QLEVEL_SCORES['recall'].append(qlevel_recall)
    #RAW_QLEVEL_SCORES['f1'].append(qlevel_f1)

In [ ]:
SCORES = {group_name: {
    metric_name: calculate_statistics(metric_scores) for metric_name, metric_scores in scores_group.items()} 
    for group_name, scores_group in [('question-level', RAW_QLEVEL_SCORES), ('searchstep-level', RAW_SLEVEL_SCORES), ('entity-level', RAW_ELEVEL_SCORES)]}

In [ ]:
with open(SAVE_NODEMATCHING_SCORES_PATH, 'w', encoding='utf-8') as fd:
    fd.write(json.dumps(SCORES, indent=2, ensure_ascii=False))

In [ ]:
SCORES